# EEG_24 — Within-Cluster Proficiency Analysis

**Domanda**: Cosa distingue un soggetto C0-proficient da uno C0-non-proficient? E C1-prof da C1-non-prof?

**Tre tipi di feature testate within-cluster vs bAcc:**
- **PCC** (1830 coppie, struttura connettività media soggetto)
- **PLV hub** (coppie chiave da EEG_23: C1↔C4 per C0, TP7↔CP5 per C1)
- **PSD** (potenza per banda per elettrodo, 5 bande × 61 = 305 feature)

**Metodi**: Spearman rho + FDR correction (Benjamini-Hochberg), Mann-Whitney top vs bottom half

**Risposta a Francesco**: i null result di EEG_16b (p=0.800) testavano C0 vs C1 → bAcc.
Questo notebook testa WITHIN cluster: dentro C0, esistono feature che separano buoni da cattivi?

## §1 — Setup

In [ ]:
import json, logging, pickle, re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.signal import butter, filtfilt, welch
from scipy.stats import mannwhitneyu, spearmanr
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg24')

project_root = Path('/home/daniele_u/miralis-hypergraph-imagined-speech')
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)

FS = 256; N_CHANNELS = 61; N_SAMPLES = 384
CLUSTER_SCHEME = 'concr4'
PLV_BANDS   = {'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 50)}
PSD_BANDS   = {'delta': (1, 4), 'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 50)}

CHAN_NAMES = ['A1','AF7','AF3','Fp1','Fp2','AF4','AF8','A2',
    'F7','F5','F3','F1','F2','F4','F6','F8',
    'FT7','FC5','FC3','FC1','FC2','FC4','FC6','FT8',
    'T7','C5','C3','C1','C2','C4','C6','T8',
    'TP7','CP5','CP3','CP1','CP2','CP4','CP6','TP8',
    'P7','P5','P3','P1','P2','P4','P6','P8',
    'FPz','PO7','PO3','O1','O2','PO4','PO8','Oz',
    'AFz','Fz','FCz','Cz','CPz']
CHAN_IDX = {n: i for i, n in enumerate(CHAN_NAMES)}

# Indici coppie triangolo superiore (1830 coppie)
PAIR_ROWS, PAIR_COLS = np.triu_indices(N_CHANNELS, k=1)
PAIR_NAMES = [f'{CHAN_NAMES[i]}-{CHAN_NAMES[j]}' for i, j in zip(PAIR_ROWS, PAIR_COLS)]
N_PAIRS = len(PAIR_ROWS)  # 1830

# Hub pairs EEG_23 — C0 (gamma motorio) e C1 (gamma temporoparietale)
C0_HUB_PAIRS = [('C1','C4'), ('CP1','Fz'), ('C1','Fz'), ('C1','FCz'), ('C1','CP1'), ('C4','CP1')]
C1_HUB_PAIRS = [('TP7','CP5'), ('TP7','P7'), ('CP5','P7'), ('T8','TP7'), ('T7','TP7'), ('P7','Cz')]

log.info(f'Setup ok. N_PAIRS={N_PAIRS}')

## §2 — Load soggetti: bAcc + cluster labels

In [ ]:
# bAcc da EEG_13b (checkpoint per soggetto)
ckpt_dir = project_root / 'models' / 'eeg13b_200e'
bacc_by_subj = {}
for p in sorted(ckpt_dir.glob('P*.pt')):
    sid = int(p.stem[1:])
    ck = torch.load(p, map_location='cpu', weights_only=False)
    for key in ['val_bacc', 'test_bacc', 'bacc', 'best_val_bacc']:
        if key in ck:
            bacc_by_subj[sid] = float(ck[key])
            break

# Cluster labels da EEG_16b
cd = json.loads((project_root / 'configs' / 'eeg16b_cluster_labels.json').read_text())
subj_cluster = {s: l for s, l in zip(cd['subj_ids'], cd['labels'])}

# DataFrame principale
df_subj = pd.DataFrame([
    {'sid': sid, 'bacc': bacc, 'cluster': subj_cluster.get(sid)}
    for sid, bacc in bacc_by_subj.items()
    if subj_cluster.get(sid) is not None
]).sort_values('sid').reset_index(drop=True)

C0_SUBJ = sorted(df_subj[df_subj.cluster==0]['sid'].tolist())
C1_SUBJ = sorted(df_subj[df_subj.cluster==1]['sid'].tolist())
ALL_SUBJ = C0_SUBJ + C1_SUBJ

log.info(f'Soggetti: {len(df_subj)}  C0={len(C0_SUBJ)}  C1={len(C1_SUBJ)}')
log.info(f'bAcc C0: {df_subj[df_subj.cluster==0]["bacc"].mean():.3f} ± {df_subj[df_subj.cluster==0]["bacc"].std():.3f}')
log.info(f'bAcc C1: {df_subj[df_subj.cluster==1]["bacc"].mean():.3f} ± {df_subj[df_subj.cluster==1]["bacc"].std():.3f}')

## §3 — Feature PCC soggetto-level (1830D)

Per ogni soggetto: media delle matrici abs_pcc su tutti i trial → vettore 1830D.

In [ ]:
_PAT = re.compile(r'^P(\d+)_S(\d+)$')
_data_root = project_root / 'data' / 'hypergraphs_pruned_abs_pcc'

def compute_subj_pcc(sid):
    """Media abs_pcc su tutti i trial del soggetto → (1830,)"""
    mats = []
    for sess_dir in sorted(_data_root.glob(f'P{sid:03d}_S*')):
        for p in sorted(sess_dir.glob('trial_*.pt')):
            d = torch.load(p, weights_only=False)
            x = d['x'].float().numpy()  # (61, 384)
            # abs_pcc
            corr = np.abs(np.corrcoef(x))  # (61, 61)
            mats.append(corr[PAIR_ROWS, PAIR_COLS])
    if not mats:
        return None
    return np.mean(mats, axis=0)  # (1830,)

log.info('Calcolo PCC per soggetto...')
PCC_BY_SUBJ = {}  # sid → (1830,)
for sid in tqdm(ALL_SUBJ, desc='PCC soggetti'):
    vec = compute_subj_pcc(sid)
    if vec is not None:
        PCC_BY_SUBJ[sid] = vec

log.info(f'PCC calcolato per {len(PCC_BY_SUBJ)} soggetti')

# Checkpoint
with open(project_root / 'data' / 'eeg24_pcc_by_subj.pkl', 'wb') as f:
    pickle.dump(PCC_BY_SUBJ, f)
log.info('Checkpoint PCC salvato.')

In [ ]:
# §3b — LOAD checkpoint (salta §3 se già eseguito)
with open(project_root / 'data' / 'eeg24_pcc_by_subj.pkl', 'rb') as f:
    PCC_BY_SUBJ = pickle.load(f)
log.info(f'Checkpoint PCC caricato: {len(PCC_BY_SUBJ)} soggetti')

## §4 — Feature PLV soggetto-level (da checkpoint EEG_23)

Riusa `eeg23_plv_records.pkl` — nessun ricalcolo.
Per ogni soggetto: media PLV su TUTTI i trial (non solo corretti) nelle hub pairs di EEG_23.

In [ ]:
CHECKPOINT_EEG23 = project_root / 'data' / 'eeg23_plv_records.pkl'

with open(CHECKPOINT_EEG23, 'rb') as f:
    plv_records = pickle.load(f)
log.info(f'EEG_23 checkpoint: {len(plv_records)} trial')

# Aggrega PLV per soggetto: media su tutti i trial
PLV_BY_SUBJ = defaultdict(lambda: defaultdict(list))  # sid → band → lista di plv_matrix
for r in plv_records:
    sid = r['subj_id']
    for band, plv_mat in r['plv'].items():
        PLV_BY_SUBJ[sid][band].append(plv_mat)

# Media per soggetto
PLV_MEAN_BY_SUBJ = {}  # sid → band → (61,61) mean PLV
for sid in PLV_BY_SUBJ:
    PLV_MEAN_BY_SUBJ[sid] = {
        band: np.mean(mats, axis=0)
        for band, mats in PLV_BY_SUBJ[sid].items()
    }

log.info(f'PLV medio calcolato per {len(PLV_MEAN_BY_SUBJ)} soggetti')

# Estrai PLV nelle hub pairs specifiche
def extract_hub_plv(sid, hub_pairs, band='gamma'):
    """Restituisce lista di valori PLV per le hub pairs di un soggetto."""
    if sid not in PLV_MEAN_BY_SUBJ:
        return None
    plv_mat = PLV_MEAN_BY_SUBJ[sid][band]
    vals = []
    for a, b in hub_pairs:
        ia, ib = CHAN_IDX.get(a), CHAN_IDX.get(b)
        if ia is not None and ib is not None:
            vals.append(plv_mat[ia, ib])
    return np.mean(vals) if vals else None

# Costruisci feature PLV per soggetto
PLV_FEATURES = []  # lista di dict per ogni soggetto
for _, row in df_subj.iterrows():
    sid = int(row['sid'])
    d = {'sid': sid, 'bacc': row['bacc'], 'cluster': int(row['cluster'])}
    for band in PLV_BANDS:
        if sid in PLV_MEAN_BY_SUBJ:
            d[f'plv_c0hub_{band}'] = extract_hub_plv(sid, C0_HUB_PAIRS, band)
            d[f'plv_c1hub_{band}'] = extract_hub_plv(sid, C1_HUB_PAIRS, band)
            # Global mean PLV (tutti i 1830 pares)
            mat = PLV_MEAN_BY_SUBJ[sid][band]
            d[f'plv_global_{band}'] = float(mat[PAIR_ROWS, PAIR_COLS].mean())
    PLV_FEATURES.append(d)

df_plv = pd.DataFrame(PLV_FEATURES)
log.info(f'PLV features: {df_plv.shape}  colonne: {[c for c in df_plv.columns if c.startswith("plv")]}')

## §5 — Feature PSD soggetto-level (5 bande × 61 canali = 305D)

Per ogni soggetto: Welch PSD media su tutti i trial → potenza media per banda per elettrodo.

In [ ]:
def compute_subj_psd(sid):
    """Media banda per banda per elettrodo su tutti i trial → dict band → (61,)"""
    band_accum = {b: [] for b in PSD_BANDS}
    for sess_dir in sorted(_data_root.glob(f'P{sid:03d}_S*')):
        for p in sorted(sess_dir.glob('trial_*.pt')):
            d = torch.load(p, weights_only=False)
            x = d['x'].float().numpy()  # (61, 384)
            freqs, psd = welch(x, fs=FS, nperseg=128, axis=1)  # (61, n_freqs)
            for band, (lo, hi) in PSD_BANDS.items():
                mask = (freqs >= lo) & (freqs <= hi)
                band_accum[band].append(psd[:, mask].mean(axis=1))  # (61,)
    if not band_accum['alpha']:
        return None
    return {band: np.mean(vals, axis=0) for band, vals in band_accum.items()}

log.info('Calcolo PSD per soggetto...')
PSD_BY_SUBJ = {}
for sid in tqdm(ALL_SUBJ, desc='PSD soggetti'):
    res = compute_subj_psd(sid)
    if res is not None:
        PSD_BY_SUBJ[sid] = res

log.info(f'PSD calcolato per {len(PSD_BY_SUBJ)} soggetti')
with open(project_root / 'data' / 'eeg24_psd_by_subj.pkl', 'wb') as f:
    pickle.dump(PSD_BY_SUBJ, f)
log.info('Checkpoint PSD salvato.')

In [ ]:
# §5b — LOAD checkpoint (salta §5 se già eseguito)
with open(project_root / 'data' / 'eeg24_psd_by_subj.pkl', 'rb') as f:
    PSD_BY_SUBJ = pickle.load(f)
log.info(f'Checkpoint PSD caricato: {len(PSD_BY_SUBJ)} soggetti')

## §6 — Analisi within-cluster: Spearman + FDR

Per ogni feature e ogni cluster: Spearman rho vs bAcc, FDR correction (Benjamini-Hochberg).

In [ ]:
def within_cluster_spearman(feature_matrix, bacc_vec, feature_names, label, alpha_fdr=0.05):
    """
    feature_matrix: (n_subj, n_feat)
    bacc_vec: (n_subj,)
    Ritorna DataFrame con rho, p_raw, p_fdr, significant per ogni feature.
    """
    n_feat = feature_matrix.shape[1]
    rhos, pvals = [], []
    for k in range(n_feat):
        rho, p = spearmanr(bacc_vec, feature_matrix[:, k])
        rhos.append(rho)
        pvals.append(p)
    
    _, p_fdr, _, _ = multipletests(pvals, method='fdr_bh')
    
    df_res = pd.DataFrame({
        'feature': feature_names,
        'rho': rhos,
        'p_raw': pvals,
        'p_fdr': p_fdr,
        'sig_raw': np.array(pvals) < 0.05,
        'sig_fdr': p_fdr < alpha_fdr,
    }).sort_values('p_raw')
    
    n_sig_raw = df_res['sig_raw'].sum()
    n_sig_fdr = df_res['sig_fdr'].sum()
    log.info(f'{label}: {n_feat} feature → {n_sig_raw} sig (p<0.05) → {n_sig_fdr} sig (FDR)')
    return df_res


def build_feature_matrix(subj_list, feature_dict, pair_rows, pair_cols):
    """Costruisce matrice (n_subj, 1830) dalle PCC medie per soggetto."""
    rows = []
    valid_sids = []
    for sid in subj_list:
        if sid in feature_dict:
            rows.append(feature_dict[sid])
            valid_sids.append(sid)
    return np.array(rows), valid_sids


RESULTS = {}  # cluster → feature_type → DataFrame risultati

for cl, cname, subj_list in [(0, 'C0', C0_SUBJ), (1, 'C1', C1_SUBJ)]:
    sub_df = df_subj[df_subj.cluster == cl]
    RESULTS[cl] = {}
    log.info(f'\n=== {cname} (n={len(sub_df)}) ===')
    
    # ── PCC ──
    mat_pcc, valid_sids = build_feature_matrix(subj_list, PCC_BY_SUBJ, PAIR_ROWS, PAIR_COLS)
    bacc_pcc = np.array([df_subj[df_subj.sid==s]['bacc'].values[0] for s in valid_sids])
    RESULTS[cl]['pcc'] = within_cluster_spearman(
        mat_pcc, bacc_pcc, PAIR_NAMES, f'{cname} PCC')
    
    # ── PSD ──
    psd_rows, psd_sids = [], []
    psd_feat_names = [f'{band}_{ch}' for band in PSD_BANDS for ch in CHAN_NAMES]
    for sid in subj_list:
        if sid in PSD_BY_SUBJ:
            row = np.concatenate([PSD_BY_SUBJ[sid][band] for band in PSD_BANDS])
            psd_rows.append(row)
            psd_sids.append(sid)
    mat_psd = np.array(psd_rows)
    bacc_psd = np.array([df_subj[df_subj.sid==s]['bacc'].values[0] for s in psd_sids])
    RESULTS[cl]['psd'] = within_cluster_spearman(
        mat_psd, bacc_psd, psd_feat_names, f'{cname} PSD')

log.info('Spearman completato.')

In [ ]:
# §6b — PLV analysis (feature scalari, non serve FDR)
plv_feat_cols = [c for c in df_plv.columns if c.startswith('plv_')]

print('\n=== PLV within-cluster Spearman ===')
for cl, cname in [(0,'C0'), (1,'C1')]:
    sub = df_plv[df_plv.cluster == cl].dropna(subset=plv_feat_cols)
    print(f'\n{cname} (n={len(sub)}):')
    print(f'  {"Feature":<30} {"rho":>8} {"p":>10}')
    print(f'  {"-"*52}')
    for feat in plv_feat_cols:
        rho, p = spearmanr(sub['bacc'], sub[feat])
        sig = ' ★' if p < 0.05 else ''
        print(f'  {feat:<30} {rho:>+8.3f} {p:>10.4f}{sig}')

## §7 — Top findings: quali coppie PCC/PSD sono significative?

Stampa top-20 coppie PCC e top-20 feature PSD per cluster (p_raw ordinato).

In [ ]:
print('\n=== TOP FINDINGS ============================')
for cl, cname in [(0,'C0'), (1,'C1')]:
    for feat_type, top_n in [('pcc', 20), ('psd', 15)]:
        df_res = RESULTS[cl][feat_type]
        n_sig_raw = df_res['sig_raw'].sum()
        n_sig_fdr = df_res['sig_fdr'].sum()
        print(f'\n{cname} {feat_type.upper()}: {n_sig_raw} sig (p<0.05), {n_sig_fdr} sig (FDR)')
        top = df_res.head(top_n)
        print(f'  {"Rank":<5} {"Feature":<30} {"rho":>8} {"p_raw":>10} {"p_fdr":>10}')
        print(f'  {"-"*68}')
        for rank, (_, row) in enumerate(top.iterrows(), 1):
            sig = ' ★FDR' if row['sig_fdr'] else (' ★' if row['sig_raw'] else '')
            print(f'  {rank:<5} {str(row["feature"]):<30} {row["rho"]:>+8.3f} {row["p_raw"]:>10.4f} {row["p_fdr"]:>10.4f}{sig}')

## §8 — Visualizzazione: volcano plot + scatter top features

In [ ]:
import matplotlib.patches as mpatches

BG = 'white'; DARK = '#1A1F2E'
C0_COLOR = '#4A90E2'; C1_COLOR = '#FF8C42'
SIG_COLOR = '#E63946'; NONSIG_COLOR = '#CCCCCC'

def volcano_plot(ax, df_res, title, color_sig):
    """Volcano plot: rho (x) vs -log10(p_raw) (y)."""
    ax.set_facecolor(BG)
    x = df_res['rho'].values
    y = -np.log10(df_res['p_raw'].values.clip(1e-300))
    sig = df_res['sig_raw'].values
    
    ax.scatter(x[~sig], y[~sig], s=6, c=NONSIG_COLOR, alpha=0.4, zorder=2)
    ax.scatter(x[sig],  y[sig],  s=12, c=color_sig,   alpha=0.8, zorder=3)
    
    ax.axhline(-np.log10(0.05), color='gray', ls='--', lw=1, alpha=0.7)
    ax.axvline(0, color='gray', ls='-', lw=0.5, alpha=0.5)
    
    n_sig = sig.sum()
    n_fdr = df_res['sig_fdr'].sum()
    ax.set_title(f'{title}\n{n_sig} sig (p<0.05) | {n_fdr} sig (FDR)', fontsize=9, color=DARK)
    ax.set_xlabel("Spearman rho", fontsize=8)
    ax.set_ylabel("-log10(p)", fontsize=8)
    ax.tick_params(colors=DARK, labelsize=7)
    
    # Annota top 5
    top5 = df_res.head(5)
    for _, row in top5.iterrows():
        xi = row['rho']; yi = -np.log10(max(row['p_raw'], 1e-300))
        ax.annotate(str(row['feature'])[:20], (xi, yi),
                    fontsize=5.5, color=DARK, ha='center', va='bottom',
                    xytext=(0, 4), textcoords='offset points')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor(BG)
fig.suptitle('EEG_24 — Within-cluster: Spearman rho vs bAcc\n'
             '(punti colorati = p<0.05 | linea tratteggiata = soglia p=0.05)',
             fontsize=12, color=DARK)

volcano_plot(axes[0,0], RESULTS[0]['pcc'], 'C0 — PCC (1830 coppie)', C0_COLOR)
volcano_plot(axes[0,1], RESULTS[1]['pcc'], 'C1 — PCC (1830 coppie)', C1_COLOR)
volcano_plot(axes[1,0], RESULTS[0]['psd'], 'C0 — PSD banda×elettrodo (305D)', C0_COLOR)
volcano_plot(axes[1,1], RESULTS[1]['psd'], 'C1 — PSD banda×elettrodo (305D)', C1_COLOR)

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg24_volcano.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show(); plt.close()
log.info('Salvato: eeg24_volcano.png')

In [ ]:
# Scatter plot: top-3 feature PCC per cluster
def scatter_top_features(df_res, feature_dict, pair_rows, pair_cols, subj_list, df_subj_in, 
                         title, color, top_n=3, fname=''):
    top_feats = df_res[df_res['sig_raw']].head(top_n)
    if len(top_feats) == 0:
        log.info(f'{title}: nessuna feature significativa')
        return
    n = len(top_feats)
    fig, axes = plt.subplots(1, n, figsize=(4.5*n, 4.5))
    if n == 1: axes = [axes]
    fig.patch.set_facecolor(BG)
    fig.suptitle(title, fontsize=10, color=DARK)
    
    for ax, (_, row) in zip(axes, top_feats.iterrows()):
        feat_name = row['feature']
        # Trova indice della feature
        feat_idx = PAIR_NAMES.index(feat_name) if feat_name in PAIR_NAMES else None
        if feat_idx is None:
            continue
        x_vals, y_vals, sids_plot = [], [], []
        for sid in subj_list:
            if sid in feature_dict:
                bacc = df_subj_in[df_subj_in.sid==sid]['bacc'].values
                if len(bacc):
                    x_vals.append(feature_dict[sid][feat_idx])
                    y_vals.append(bacc[0])
                    sids_plot.append(sid)
        ax.set_facecolor(BG)
        ax.scatter(x_vals, y_vals, s=55, c=color, alpha=0.7,
                   edgecolors='white', lw=0.5, zorder=3)
        # Trend line
        m, b = np.polyfit(x_vals, y_vals, 1)
        xl = np.linspace(min(x_vals), max(x_vals), 50)
        ax.plot(xl, m*xl+b, color=color, lw=1.5, alpha=0.6)
        ax.axhline(0.25, color='gray', ls='--', lw=1, alpha=0.6)
        ax.set_xlabel(f'PCC {feat_name}', fontsize=8)
        ax.set_ylabel('bAcc (EEG_13b)', fontsize=8)
        ax.set_title(f'rho={row["rho"]:+.3f}  p={row["p_raw"]:.4f}', fontsize=9, color=DARK)
        ax.tick_params(colors=DARK, labelsize=7)
    
    plt.tight_layout()
    if fname:
        plt.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight', facecolor=BG)
        log.info(f'Salvato: {fname}')
    plt.show(); plt.close()

scatter_top_features(RESULTS[0]['pcc'], PCC_BY_SUBJ, PAIR_ROWS, PAIR_COLS,
    C0_SUBJ, df_subj, 'C0 — Top PCC features vs bAcc', C0_COLOR,
    fname='eeg24_C0_pcc_scatter.png')
scatter_top_features(RESULTS[1]['pcc'], PCC_BY_SUBJ, PAIR_ROWS, PAIR_COLS,
    C1_SUBJ, df_subj, 'C1 — Top PCC features vs bAcc', C1_COLOR,
    fname='eeg24_C1_pcc_scatter.png')

In [ ]:
# Scatter PLV hub pairs vs bAcc
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.patch.set_facecolor(BG)
fig.suptitle('EEG_24 — PLV hub pairs (EEG_23) vs bAcc per cluster', fontsize=11, color=DARK)

hub_feats_to_plot = [
    ('plv_c0hub_gamma', 'C0 hub γ\n(C1↔C4,CP1↔Fz)', C0_COLOR),
    ('plv_c0hub_alpha', 'C0 hub α', C0_COLOR),
    ('plv_global_gamma', 'Global γ PLV', '#888888'),
    ('plv_c1hub_gamma', 'C1 hub γ\n(TP7↔CP5,P7)', C1_COLOR),
    ('plv_c1hub_beta',  'C1 hub β', C1_COLOR),
    ('plv_global_alpha', 'Global α PLV', '#888888'),
]

for ax, (feat, label, col) in zip(axes.flat, hub_feats_to_plot):
    ax.set_facecolor(BG)
    for cl, cname, marker, lw in [(0,'C0','o',1.5), (1,'C1','s',1.5)]:
        sub = df_plv[(df_plv.cluster==cl) & df_plv[feat].notna()]
        if len(sub) < 3: continue
        color_cl = C0_COLOR if cl==0 else C1_COLOR
        ax.scatter(sub[feat], sub['bacc'], s=45, c=color_cl,
                   marker=marker, alpha=0.7, edgecolors='white', lw=0.4,
                   label=cname, zorder=3)
        rho, p = spearmanr(sub['bacc'], sub[feat])
        m, b = np.polyfit(sub[feat], sub['bacc'], 1)
        xl = np.linspace(sub[feat].min(), sub[feat].max(), 50)
        ax.plot(xl, m*xl+b, color=color_cl, lw=1.3, alpha=0.5)
    
    ax.axhline(0.25, color='gray', ls='--', lw=0.8, alpha=0.6)
    ax.set_xlabel(label, fontsize=8)
    ax.set_ylabel('bAcc', fontsize=8)
    ax.legend(fontsize=7, framealpha=0.7)
    ax.tick_params(colors=DARK, labelsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor('#DDDDDD')

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg24_plv_scatter.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show(); plt.close()
log.info('Salvato: eeg24_plv_scatter.png')

## §9 — Verdict

Riepilogo completo: quante feature significative (p<0.05 e FDR) per ogni cluster e tipo.

In [ ]:
print('=' * 65)
print('EEG_24 — VERDICT: Within-cluster proficiency predictors')
print('=' * 65)

for cl, cname in [(0,'C0'), (1,'C1')]:
    sub = df_subj[df_subj.cluster==cl]
    print(f'\n{cname} (n={len(sub)}  bAcc={sub["bacc"].mean():.3f}±{sub["bacc"].std():.3f})')
    for feat_type in ['pcc', 'psd']:
        df_res = RESULTS[cl][feat_type]
        n_total = len(df_res)
        n_sig_raw = df_res['sig_raw'].sum()
        n_sig_fdr = df_res['sig_fdr'].sum()
        top1 = df_res.iloc[0]
        print(f'  {feat_type.upper()} ({n_total} feat): {n_sig_raw} sig p<0.05 → {n_sig_fdr} FDR')
        print(f'    top1: {top1["feature"]}  rho={top1["rho"]:+.3f}  p={top1["p_raw"]:.4f}')

print(f'\nPLV hub pairs:')
for cl, cname in [(0,'C0'), (1,'C1')]:
    sub = df_plv[df_plv.cluster==cl]
    print(f'  {cname}:')
    for feat in plv_feat_cols:
        s2 = sub.dropna(subset=[feat])
        if len(s2) < 5: continue
        rho, p = spearmanr(s2['bacc'], s2[feat])
        sig = ' ★' if p < 0.05 else ''
        print(f'    {feat:<30} rho={rho:+.3f}  p={p:.4f}{sig}')

print('\n' + '=' * 65)
print('INTERPRETAZIONE:')
print('  Se 0 feature FDR → null result n.4: nessuna feature soggetto')
print('  predice proficiency within-cluster (confermato esaurimento).')
print('  Se feature FDR significative → nuovo finding per la tesi.')
print('=' * 65)